# YUYEONG 기술스택과 Agent 설계

이 Notebook은 YUYEONG 포트폴리오의 기술스택, Agent 파이프라인, 저장소 구조와 안전 검증 항목을 재현 가능한 형태로 정리합니다.

**Pipeline:** 기획 → 수립 → 기술스택·실행 → 수정 → 회고

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
if not (ROOT / 'package.json').exists():
    ROOT = ROOT.parent

print(f'Project root: {ROOT.resolve()}')

## 1. 기술스택

프로젝트의 실제 `package.json`과 소스 구현을 기준으로 작성했습니다. 외부 Python 패키지는 필요하지 않습니다.

In [ ]:
STACK = [
    {'layer': 'Language', 'technology': 'TypeScript 5.9', 'purpose': '엄격한 타입 기반 UI·API 구현'},
    {'layer': 'UI', 'technology': 'React 19.2', 'purpose': 'Agent 콘솔, 후보 보드, 동선과 메뉴'},
    {'layer': 'Framework', 'technology': 'Next.js 16.2 + vinext', 'purpose': 'App Router와 Cloudflare 호환 빌드'},
    {'layer': 'Build', 'technology': 'Vite 8', 'purpose': '개발 서버와 프로덕션 번들'},
    {'layer': 'LLM', 'technology': 'GPT-5.6 Sol', 'purpose': '최종 제약·근거 안전 요약'},
    {'layer': 'LLM API', 'technology': 'OpenAI Responses API', 'purpose': '서버 측 모델 호출'},
    {'layer': 'Place', 'technology': 'OpenStreetMap + Overpass', 'purpose': '식당·카페·드라이브 후보'},
    {'layer': 'Geocoding', 'technology': 'Nominatim', 'purpose': '장소 문자열을 좌표로 변환'},
    {'layer': 'Weather', 'technology': 'Open-Meteo', 'purpose': '현재 날씨와 강수 조건'},
    {'layer': 'Routing', 'technology': 'OSRM Route + Table', 'purpose': '전체 동선과 20분 드라이브 검증'},
    {'layer': 'Hosting', 'technology': 'OpenAI Sites', 'purpose': '비공개 프로덕션과 비밀 변수'},
    {'layer': 'Test', 'technology': 'Node.js test runner', 'purpose': '렌더링·정책·기능 회귀 테스트'},
]

widths = {key: max(len(key), *(len(str(row[key])) for row in STACK)) for key in STACK[0]}
header = ' | '.join(key.upper().ljust(widths[key]) for key in widths)
line = '-+-'.join('-' * widths[key] for key in widths)
print(header)
print(line)
for row in STACK:
    print(' | '.join(str(row[key]).ljust(widths[key]) for key in widths))

## 2. Agent 실행 파이프라인

1. **REQUEST** — 위치, 시간, 동행, 분위기, 활동, 식사 예산 정규화  
2. **GROUND** — Nominatim, Open-Meteo, Overpass, OSRM에서 사실 수집  
3. **FILTER** — 이름·좌표 확인, 중복 제거, 카테고리별 후보 4곳 검증  
4. **PLAN** — 점심 → 카페 → 저녁 → 드라이브 동선 생성  
5. **VERIFY** — 경로 구간, 종료시간, 예산 범위, 드라이브 20분 제약 검산  
6. **REPORT** — FACT, INFERENCE, UNKNOWN을 분리해 응답

내부 Chain-of-Thought는 공개하지 않고 ReAct의 Action, Observation, Replan과 검산 결과만 보여줍니다.

In [ ]:
PIPELINE = {
    'input': ['location', 'time window', 'companion', 'mood', 'activities', 'restaurant budget'],
    'tools': ['Location', 'Weather', 'Places', 'CandidateExpand', 'Rank', 'DriveConstraint', 'Route'],
    'reasoning': ['bounded ReAct', 'six-stage audit', 'deterministic hard constraints'],
    'critic': {'model': 'gpt-5.6-sol', 'endpoint': '/v1/responses'},
    'output': ['4 restaurants', '4 cafes', '4 drive courses >= 20 min', 'final itinerary', 'menus', 'evidence'],
}
print(json.dumps(PIPELINE, ensure_ascii=False, indent=2))

## 3. 저장소 구조 점검

In [ ]:
REQUIRED_FILES = [
    'README.md',
    'package.json',
    '.env.example',
    'app/ExperienceAgent.tsx',
    'app/agent/model.ts',
    'app/agent/system-prompt.ts',
    'app/api/experience/route.ts',
    'tests/rendered-html.test.mjs',
    'public/og.png',
]

checks = {path: (ROOT / path).exists() for path in REQUIRED_FILES}
for path, exists in checks.items():
    print(f"{'PASS' if exists else 'FAIL'}  {path}")

assert all(checks.values()), '필수 포트폴리오 파일이 누락되었습니다.'

## 4. 핵심 기능 계약 점검

외부 API를 호출하지 않고 소스에 핵심 안전 계약이 존재하는지 확인합니다.

In [ ]:
route_source = (ROOT / 'app/api/experience/route.ts').read_text(encoding='utf-8')
prompt_source = (ROOT / 'app/agent/system-prompt.ts').read_text(encoding='utf-8')
model_source = (ROOT / 'app/agent/model.ts').read_text(encoding='utf-8')
ui_source = (ROOT / 'app/ExperienceAgent.tsx').read_text(encoding='utf-8')

CONTRACTS = {
    'GPT-5.6 Sol model id': 'gpt-5.6-sol' in model_source,
    'Responses API': 'api.openai.com/v1/responses' in model_source,
    'Private reasoning policy': 'Do not expose raw chain-of-thought' in prompt_source,
    'Bounded ReAct': 'bounded ReAct loop' in prompt_source,
    'Four candidates': 'RECOMMENDATION_LIMIT = 4' in route_source,
    'Drive minimum 20 minutes': 'MIN_DRIVE_MINUTES = 20' in route_source,
    'OSRM Table service': 'table/v1/driving' in route_source,
    'Restaurant-only budget': 'Never apply it to cafes' in prompt_source,
    'Candidate board UI': '4 PER CATEGORY' in ui_source,
}

for name, passed in CONTRACTS.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")

assert all(CONTRACTS.values()), '핵심 기능 계약을 다시 확인하세요.'

## 5. 보안 점검

이 Notebook은 `.env`의 값을 읽거나 출력하지 않습니다. GitHub에는 `.env.example`만 포함하고 실제 API key는 서버 비밀 변수로 관리해야 합니다.

In [ ]:
gitignore = (ROOT / '.gitignore').read_text(encoding='utf-8')
security_checks = {
    'env files ignored': '.env*' in gitignore,
    'example env allowed': '!.env.example' in gitignore,
    'example contains placeholder': 'your_openai_api_key' in (ROOT / '.env.example').read_text(encoding='utf-8'),
}
for name, passed in security_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")
assert all(security_checks.values())

## 6. 실행과 검증

```bash
npm install
npm run dev
npm test
```

모델을 활성화하려면 `.env.local` 또는 배포 환경의 암호화된 비밀 변수에 `OPENAI_API_KEY`를 설정합니다.

## 7. 회고 요약

- LLM은 설명과 최종 비평을 담당하고, 장소 수·중복·경로·20분 제약은 코드가 보장합니다.
- 외부 데이터가 없을 때는 추측하지 않고 UNKNOWN 또는 fallback을 반환합니다.
- 사용자 피드백은 UI 문구뿐 아니라 정책, 도구, 검증, 테스트까지 함께 수정했습니다.
- 다음 단계는 후보 선택 후 즉시 재계획하는 Human-in-the-loop와 실제 메뉴·예약 API입니다.